In [9]:
# ─────────────────────────────────────────────────────────────────────────────
# 1) Imports, config & data loading
# ─────────────────────────────────────────────────────────────────────────────
import os, time
import warnings
import numpy as np
import pandas as pd
import joblib
from sklearn.metrics import mean_squared_error, r2_score

import xgboost as xgb
# silence all xgboost logs
xgb.set_config(verbosity=0)
warnings.filterwarnings("ignore")

# load your splits
X_tr, y_tr = joblib.load("data/processed/train.pkl")
X_val, y_val = joblib.load("data/processed/val.pkl")
X_te, y_te   = joblib.load("data/processed/test.pkl")

cfg = {
    "general": {
        "n_trials": 50,
        "seeds":    [0, 42, 7, 123, 999],
    },
    "xgb": {
        "n_estimators":          1000,
        "early_stopping_rounds": 50,
        "search_space": {
            "learning_rate":   {"low": 1e-3, "high": 1e-1},
            "max_depth":       {"low": 3,    "high": 12},
            "subsample":       {"low": 0.5,  "high": 1.0},
            "colsample_bytree":{"low": 0.5,  "high": 1.0},
        },
    },
}

In [11]:
# ─────────────────────────────────────────────────────────────────────────────
# 2) Three XGBoost‐tuning functions w/ Optuna, Hyperopt, skopt
# ─────────────────────────────────────────────────────────────────────────────
import optuna
from hyperopt import fmin, tpe, hp, Trials, STATUS_OK
from skopt import gp_minimize
from skopt.space import Real, Integer

def tune_optuna_xgb(seed):
    def objective(trial):
        params = {
            "objective":       "reg:squarederror",
            "eval_metric":     "rmse",
            "seed":            seed,
            "eta":             trial.suggest_float(
                                   "learning_rate",
                                   **cfg["xgb"]["search_space"]["learning_rate"],
                                   log=True
                               ),
            "max_depth":       trial.suggest_int(
                                   "max_depth",
                                   **cfg["xgb"]["search_space"]["max_depth"]
                               ),
            "subsample":       trial.suggest_float(
                                   "subsample",
                                   **cfg["xgb"]["search_space"]["subsample"]
                               ),
            "colsample_bytree":trial.suggest_float(
                                   "colsample_bytree",
                                   **cfg["xgb"]["search_space"]["colsample_bytree"]
                               ),
        }
        dtrain = xgb.DMatrix(X_tr, label=y_tr)
        dval   = xgb.DMatrix(X_val, label=y_val)
        bst = xgb.train(
            params,
            dtrain,
            num_boost_round=cfg["xgb"]["n_estimators"],
            evals=[(dval, "eval")],
            early_stopping_rounds=cfg["xgb"]["early_stopping_rounds"],
            verbose_eval=False,
        )
        preds = bst.predict(
            dval,
            iteration_range=(0, bst.best_iteration)
        )
        return mean_squared_error(y_val, preds, squared=False)

    study = optuna.create_study(
        direction="minimize",
        sampler=optuna.samplers.TPESampler(seed=seed)
    )
    study.optimize(objective, n_trials=cfg["general"]["n_trials"])
    best_p = study.best_params; best_p["seed"]=seed
    return best_p, study.best_value


def tune_hyperopt_xgb(seed):
    space = {
        "learning_rate":   hp.loguniform(
                                "learning_rate",
                                np.log(cfg["xgb"]["search_space"]["learning_rate"]["low"]),
                                np.log(cfg["xgb"]["search_space"]["learning_rate"]["high"])
                           ),
        "max_depth":       hp.quniform(
                                "max_depth",
                                cfg["xgb"]["search_space"]["max_depth"]["low"],
                                cfg["xgb"]["search_space"]["max_depth"]["high"],
                                1
                           ),
        "subsample":       hp.uniform(
                                "subsample",
                                cfg["xgb"]["search_space"]["subsample"]["low"],
                                cfg["xgb"]["search_space"]["subsample"]["high"]
                           ),
        "colsample_bytree":hp.uniform(
                                "colsample_bytree",
                                cfg["xgb"]["search_space"]["colsample_bytree"]["low"],
                                cfg["xgb"]["search_space"]["colsample_bytree"]["high"]
                           ),
    }
    trials = Trials()
    def objective(params):
        p = {
            "objective":       "reg:squarederror",
            "eval_metric":     "rmse",
            "seed":            seed,
            "eta":             params["learning_rate"],
            "max_depth":       int(params["max_depth"]),
            "subsample":       params["subsample"],
            "colsample_bytree":params["colsample_bytree"],
        }
        dtrain = xgb.DMatrix(X_tr, label=y_tr)
        dval   = xgb.DMatrix(X_val, label=y_val)
        bst = xgb.train(
            p, dtrain,
            num_boost_round=cfg["xgb"]["n_estimators"],
            evals=[(dval, "eval")],
            early_stopping_rounds=cfg["xgb"]["early_stopping_rounds"],
            verbose_eval=False,
        )
        preds = bst.predict(
            dval,
            iteration_range=(0, bst.best_iteration)
        )
        return {"loss": mean_squared_error(y_val, preds, squared=False),
                "status": STATUS_OK}

    best = fmin(
        fn=objective,
        space=space,
        algo=tpe.suggest,
        max_evals=cfg["general"]["n_trials"],
        trials=trials,
        rstate=np.random.default_rng(seed)
    )
    best_rmse = np.min(trials.losses())
    best_params = {
        "learning_rate":   best["learning_rate"],
        "max_depth":       int(best["max_depth"]),
        "subsample":       best["subsample"],
        "colsample_bytree":best["colsample_bytree"],
        "seed":            seed,
    }
    return best_params, best_rmse


def tune_skopt_xgb(seed):
    dims = [
        Real(cfg["xgb"]["search_space"]["learning_rate"]["low"],
             cfg["xgb"]["search_space"]["learning_rate"]["high"],
             prior="log-uniform", name="learning_rate"),
        Integer(cfg["xgb"]["search_space"]["max_depth"]["low"],
                cfg["xgb"]["search_space"]["max_depth"]["high"],
                name="max_depth"),
        Real(cfg["xgb"]["search_space"]["subsample"]["low"],
             cfg["xgb"]["search_space"]["subsample"]["high"],
             name="subsample"),
        Real(cfg["xgb"]["search_space"]["colsample_bytree"]["low"],
             cfg["xgb"]["search_space"]["colsample_bytree"]["high"],
             name="colsample_bytree"),
    ]
    def objective(x):
        p = {
            "objective":       "reg:squarederror",
            "eval_metric":     "rmse",
            "seed":            seed,
            "eta":             x[0],
            "max_depth":       int(x[1]),
            "subsample":       x[2],
            "colsample_bytree":x[3],
        }
        dtrain = xgb.DMatrix(X_tr, label=y_tr)
        dval   = xgb.DMatrix(X_val, label=y_val)
        bst = xgb.train(
            p, dtrain,
            num_boost_round=cfg["xgb"]["n_estimators"],
            evals=[(dval, "eval")],
            early_stopping_rounds=cfg["xgb"]["early_stopping_rounds"],
            verbose_eval=False,
        )
        preds = bst.predict(
            dval,
            iteration_range=(0, bst.best_iteration)
        )
        return mean_squared_error(y_val, preds, squared=False)

    res = gp_minimize(
        objective,
        dims,
        n_calls=cfg["general"]["n_trials"],
        random_state=seed
    )
    best_params = {
        "learning_rate":   res.x[0],
        "max_depth":       int(res.x[1]),
        "subsample":       res.x[2],
        "colsample_bytree":res.x[3],
        "seed":            seed,
    }
    return best_params, res.fun

In [13]:
# ─────────────────────────────────────────────────────────────────────────────
# 3) Run all 3 tuners → save rq1_xgb_tuning.csv
# ─────────────────────────────────────────────────────────────────────────────
methods = [
    ("Optuna",   tune_optuna_xgb),
    ("Hyperopt", tune_hyperopt_xgb),
    ("Skopt",    tune_skopt_xgb),
]

records = []
for pkg, fn in methods:
    for sd in cfg["general"]["seeds"]:
        p, rmse = fn(sd)
        records.append({
            "package":  pkg,
            "seed":     sd,
            "val_rmse": rmse,
            **p
        })

df_tune = pd.DataFrame(records)
df_tune.to_csv("rq1_xgb_tuning.csv", index=False)
print("→ tuning results → rq1_xgb_tuning.csv")
df_tune

[I 2025-06-09 21:17:40,000] A new study created in memory with name: no-name-8c8ea234-647b-46bb-8866-e56a5039528c
[I 2025-06-09 21:17:49,704] Trial 0 finished with value: 8641.388565814732 and parameters: {'learning_rate': 0.012520653814999465, 'max_depth': 10, 'subsample': 0.801381688035822, 'colsample_bytree': 0.7724415914984484}. Best is trial 0 with value: 8641.388565814732.
[I 2025-06-09 21:17:58,567] Trial 1 finished with value: 8834.47645338759 and parameters: {'learning_rate': 0.00703573702872215, 'max_depth': 9, 'subsample': 0.7187936056313462, 'colsample_bytree': 0.9458865003910399}. Best is trial 0 with value: 8641.388565814732.
[I 2025-06-09 21:18:01,123] Trial 2 finished with value: 7828.692636977503 and parameters: {'learning_rate': 0.08459126528049374, 'max_depth': 6, 'subsample': 0.8958625190413323, 'colsample_bytree': 0.7644474598764522}. Best is trial 2 with value: 7828.692636977503.
[I 2025-06-09 21:18:11,928] Trial 3 finished with value: 9164.026617014577 and parame

100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 50/50 [05:12<00:00,  6.24s/trial, best loss: 7719.736061173959]
→ tuning results → rq1_xgb_tuning.csv


,package,seed,val_rmse,learning_rate,max_depth,subsample,colsample_bytree
0,Optuna,0,7558.047024,0.030926,5,0.955764,0.761979
1,Optuna,42,7566.116810,0.067745,5,0.998311,0.812662
2,Optuna,7,7646.792763,0.037145,5,0.886911,0.650480
3,Optuna,123,7526.869004,0.077387,6,0.978564,0.878351
4,Optuna,999,7422.765119,0.098776,5,0.993475,0.576062
5,Hyperopt,0,7662.295158,0.063297,6,0.838451,0.761423
6,Hyperopt,42,7629.861490,0.097934,4,0.923423,0.970962
7,Hyperopt,7,7797.638112,0.032895,5,0.550268,0.878993
8,Hyperopt,123,7645.283797,0.040443,6,0.958807,0.751944
9,Hyperopt,999,7719.736061,0.039365,5,0.895050,0.729606


In [16]:
# ─────────────────────────────────────────────────────────────────────────────
# 4) Pick best, retrain on train+val & test → save rq1_xgb_final.csv
# ─────────────────────────────────────────────────────────────────────────────
best = df_tune.loc[df_tune.val_rmse.idxmin()]
params = {
    "objective":       "reg:squarederror",
    "eval_metric":     "rmse",
    "eta":             best.learning_rate,
    "max_depth":       int(best.max_depth),
    "subsample":       best.subsample,
    "colsample_bytree":best.colsample_bytree,
    "seed":            int(best.seed),
}

# full DMatrix
X_full = np.vstack([X_tr, X_val])
y_full = np.concatenate([y_tr, y_val])
dtrain = xgb.DMatrix(X_full, label=y_full)
dtest  = xgb.DMatrix(X_te,  label=y_te)

watch = [(dtrain, "train"), (dtest, "eval")]

t0 = time.time()
bst = xgb.train(
    params,
    dtrain,
    num_boost_round=cfg["xgb"]["n_estimators"],
    evals=watch,
    early_stopping_rounds=cfg["xgb"]["early_stopping_rounds"],
    verbose_eval=False,
)
train_time = time.time() - t0

t1 = time.time()
y_pred = bst.predict(
    dtest,
    iteration_range=(0, bst.best_iteration)
)
pred_time = time.time() - t1

test_rmse = mean_squared_error(y_te, y_pred, squared=False)
test_r2   = r2_score(y_te, y_pred)

df_final = pd.DataFrame([{
    **params,
    "n_estimators":    bst.best_iteration,
    "train_time_s":    round(train_time,3),
    "pred_time_s":     round(pred_time,3),
    "test_rmse":       round(test_rmse,3),
    "test_r2":         round(test_r2,4),
}])
df_final.to_csv("rq1_xgb_final.csv", index=False)
print("→ final results → rq1_xgb_final.csv")
df_final

→ final results → rq1_xgb_final.csv


,objective,eval_metric,eta,max_depth,subsample,colsample_bytree,seed,n_estimators,train_time_s,pred_time_s,test_rmse,test_r2
0,reg:squarederror,rmse,0.098776,5,0.993475,0.576062,999,356,3.288,0.032,6449.413,0.926
